## XGBoost Regression 

In [1]:
## Import requirement Library 
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot  as plt 
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [2]:
data=pd.read_csv('cardekho_imputated.csv')
data.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [3]:
data.shape

(15411, 14)

In [4]:
data.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [5]:
## Remove unnessesary  columns 
data.drop('Unnamed: 0',axis=1,inplace=True)
data.drop('car_name',axis=1,inplace=True)

In [6]:

col=data.columns
categorical_cols=[]
categorical_cols.clear()
for i in col:
    if data[i].dtypes=='object':
        categorical_cols.append(i)

In [7]:
categorical_cols

['brand', 'model', 'seller_type', 'fuel_type', 'transmission_type']

In [8]:

for i in categorical_cols:
    print(i,' = ',len(data[i].unique()))
    

brand  =  32
model  =  120
seller_type  =  3
fuel_type  =  5
transmission_type  =  2


In [9]:
data.head()

,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [10]:
data.shape

(15411, 12)

In [11]:
## Remove the Brand 
data.drop('brand',axis=1,inplace=True)

In [12]:
data.head()
df=data

In [13]:
## Getting ALl Different Types Of Features 
num_features=[feature for feature in df.columns if df[feature].dtype != 'O']
print('Num of Numerical Features : ' , len(num_features))
cat_features=[feature for feature in df.columns if df[feature].dtype == 'O']
print('Num of Categorical Features : ', len(cat_features))
discrete_features=[feature for feature in num_features if len(df[feature].unique())<=25]
print('Num of Discrete Features : ' , len(discrete_features))
continuous_features=[feature for feature in num_features if feature not in discrete_features]
print('Num of Continuous Features : ' , len(continuous_features))

Num of Numerical Features :  7
Num of Categorical Features :  4
Num of Discrete Features :  2
Num of Continuous Features :  5


In [14]:
## encoding the data in the categorical field 
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [15]:
## Split the data into train and test set 
x=data.drop('selling_price',axis=1)
y=data['selling_price']
x.head()


,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [16]:
from sklearn.preprocessing import LabelEncoder 
le=LabelEncoder()
x['model']=le.fit_transform(x['model'])
x['seller_type']=le.fit_transform(x['seller_type'])
x['fuel_type']=le.fit_transform(x['fuel_type'])
x['transmission_type']=le.fit_transform(x['transmission_type'])

In [17]:
x.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,1,4,1,19.70,796,46.30,5
1,54,5,20000,1,4,1,18.90,1197,82.00,5
2,118,11,60000,1,4,1,17.00,1197,80.00,5
3,7,9,37000,1,4,1,20.92,998,67.10,5
4,38,6,30000,0,1,1,22.77,1498,98.59,5


In [18]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)

In [19]:
x_train.shape,x_test.shape,y_train.shape,y_test.shape

((10787, 10), (4624, 10), (10787,), (4624,))

In [20]:
from sklearn.ensemble import RandomForestRegressor


In [21]:
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error,r2_score
from xgboost import XGBRegressor


In [22]:
models={
    'Random Forest : ':RandomForestRegressor(),
    'XG Boost : ' : XGBRegressor()
}

In [23]:
for i in range(len(list(models))):
    model=list(models.values())[i]
    name=list(models.keys())[i]

    model.fit(x_train,y_train)

    y_pred_train=model.predict(x_train)
    y_pred_test=model.predict(x_test)

    r2=r2_score(y_pred_train,y_train)
    mae=mean_absolute_error(y_pred_train,y_train)
    mse=mean_squared_error(y_pred_train,y_train)

    r2t=r2_score(y_pred_test,y_test)
    maet=mean_absolute_error(y_pred_test,y_test)
    mset=mean_squared_error(y_pred_test,y_test)
    
    print('='*40,'Training ',name,'='*40)
    print('r2_score : ',r2)
    print('Mae : ',mae)
    print('MSE : ', mse)

    print('='*40,'Testing ',name,'='*40)
    print('r2_score : ',r2t)
    print('Mae : ',maet)
    print('MSE : ', mset)
    print('='*30  , "Accuracy Difference training and testing : ",round(r2-r2t,2)*100 , '='*30)

======================================== Training  Random Forest :  ========================================
r2_score :  0.9749052743490535
Mae :  39984.67720193499
MSE :  18275848037.645996
======================================== Testing  Random Forest :  ========================================
r2_score :  0.9154319973037258
Mae :  105124.06613496557
MSE :  59063566117.78154
============================== Accuracy Difference training and testing :  0.05947327704532768 ==============================
======================================== Training  XG Boost :  ========================================
r2_score :  0.9911468625068665
Mae :  59888.0390625
MSE :  7138267648.0
======================================== Testing  XG Boost :  ========================================
r2_score :  0.8980504274368286
Mae :  102860.2578125
MSE :  80163946496.0
============================== Accuracy Difference training and testing :  0.09309643507003784 ==============================


In [24]:
# Hyperparameter tuning 
from sklearn.model_selection import RandomizedSearchCV


In [25]:
rf_params={'max_depth':[5,8,15,None,10],
           'max_features':[5,7,'auto',8],
           'min_samples_split':[2,8,15,20],
           'n_estimators':[100,200,500,1000]}
xg_params={
    "n_estimators": [100, 200, 300, 500, 800, 1000],
    "max_depth": [3, 4, 5, 6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.2, 0.3, 0.5, 1],
    "min_child_weight": [1, 3, 5, 7, 10],
    "reg_alpha": [0, 0.1, 0.5, 1],
    "reg_lambda": [0.5, 1, 1.5, 2, 5],
    "max_delta_step": [0, 1, 5, 10]
}


In [26]:
randomcv_models=[('Random Forest : ',RandomForestRegressor(),rf_params),
                 ('XGBoost : '  , XGBRegressor(),xg_params)
                ]


In [28]:
# Hyperparameter Tuning 
from sklearn.model_selection import RandomizedSearchCV

model_param={}
for name,model,params in randomcv_models:
    random=RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=100,
        cv=3,
        verbose=2,
        n_jobs=1
    )
    random.fit(x_train,y_train)
    model_param[name]=random.best_params_

for model_name in model_param:
    print(f'---------------------Best Params For     {model_name}-----------------')
    print(model_param[model_name])
    

Fitting 3 folds for each of 100 candidates, totalling 300 fits
[CV] END max_depth=10, max_features=7, min_samples_split=2, n_estimators=100; total time=   3.3s
[CV] END max_depth=10, max_features=7, min_samples_split=2, n_estimators=100; total time=   3.3s
[CV] END max_depth=10, max_features=7, min_samples_split=2, n_estimators=100; total time=   3.3s
[CV] END max_depth=None, max_features=5, min_samples_split=2, n_estimators=100; total time=   5.0s
[CV] END max_depth=None, max_features=5, min_samples_split=2, n_estimators=100; total time=   4.7s
[CV] END max_depth=None, max_features=5, min_samples_split=2, n_estimators=100; total time=   4.7s
[CV] END max_depth=15, max_features=auto, min_samples_split=8, n_estimators=200; total time=   0.0s
[CV] END max_depth=15, max_features=auto, min_samples_split=8, n_estimators=200; total time=   0.0s
[CV] END max_depth=15, max_features=auto, min_samples_split=8, n_estimators=200; total time=   0.0s
[CV] END max_depth=8, max_features=8, min_samples

In [29]:
models={
    'Random Forest  ' : RandomForestRegressor(
        n_estimators=100,min_samples_split=2,max_features=5,max_depth=None
    ),
    'XG boost ': XGBRegressor(subsample=0.9,reg_lambda=1,reg_alpha=0.1,n_estimators=200,min_child_weight=10,max_depth=5,max_delta_step=0,learning_rate=0.2,gamma=0.2,colsample_bytree=0.8)
}

In [30]:
models

{'Random Forest  ': RandomForestRegressor(max_features=5),
 'XG boost ': XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=0.2, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.2, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=0, max_depth=5,
              max_leaves=None, min_child_weight=10, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)}

In [31]:
for i in range(len(list(models))):
    model=list(models.values())[i]
    name=list(models.keys())[i]

    model.fit(x_train,y_train)

    y_pred_train=model.predict(x_train)
    y_pred_test=model.predict(x_test)

    r2=r2_score(y_pred_train,y_train)
    mae=mean_absolute_error(y_pred_train,y_train)
    mse=mean_squared_error(y_pred_train,y_train)

    r2t=r2_score(y_pred_test,y_test)
    maet=mean_absolute_error(y_pred_test,y_test)
    mset=mean_squared_error(y_pred_test,y_test)
    
    print('='*40,'Training ',name,'='*40)
    print('r2_score : ',r2)
    print('Mae : ',mae)
    print('MSE : ', mse)

    print('='*40,'Testing ',name,'='*40)
    print('r2_score : ',r2t)
    print('Mae : ',maet)
    print('MSE : ', mset)
    print('='*30  , "Accuracy Difference training and testing : ",round(r2-r2t,2)*100 , '='*30)

======================================== Training  Random Forest   ========================================
r2_score :  0.9719776899769839
Mae :  39288.48511529363
MSE :  19754431871.064255
======================================== Testing  Random Forest   ========================================
r2_score :  0.9191575243517314
Mae :  102430.09347004518
MSE :  54215835672.15968
============================== Accuracy Difference training and testing :  5.0 ==============================
======================================== Training  XG boost  ========================================
r2_score :  0.9700230956077576
Mae :  88221.65625
MSE :  22864105472.0
======================================== Testing  XG boost  ========================================
r2_score :  0.8723728060722351
Mae :  115239.1796875
MSE :  107700232192.0
============================== Accuracy Difference training and testing :  10.0 ==============================


In [32]:
data.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [35]:
data.to_csv('Clean_data.csv',index=False)